# Modbus/TCP Anomaly Detection Feature Extraction

This notebook extracts Modbus/TCP packets from pcap files and creates a CSV dataset optimized for anomaly detection.

In [ ]:
import pyshark
import csv
import os
from datetime import datetime

In [ ]:
def extract_modbus_to_csv(pcap_file, output_csv):
    """
    Extract Modbus/TCP packets with fields relevant for anomaly detection.
    
    Args:
        pcap_file: Path to the pcap file
        output_csv: Output CSV filename
    """
    # Open the pcap file
    print(f"Opening pcap file: {pcap_file}")
    cap = pyshark.FileCapture(pcap_file, display_filter='modbus')
    
    # Define CSV headers optimized for anomaly detection
    headers = [
        'packet_num',
        'timestamp',
        'time_delta',              # Time since last packet (for rate analysis)
        'src_ip',
        'dst_ip',
        'src_port',
        'dst_port',
        'tcp_len',                 # TCP payload length
        'transaction_id',
        'protocol_id',
        'modbus_length',           # Modbus message length
        'unit_id',
        'function_code',
        'is_response',             # Request (0) or Response (1)
        'is_exception',            # Exception response flag
        'exception_code',          # Exception code if applicable
        'reference_num',           # Starting address
        'word_count',              # Number of registers/coils
        'byte_count',              # Number of bytes
        'register_values',         # Actual data values (for value-based anomalies)
        'num_values',              # Count of data values
    ]
    
    modbus_packets = []
    packet_count = 0
    last_timestamp = None
    
    print("Extracting Modbus/TCP packets for anomaly detection...")
    
    try:
        for packet in cap:
            packet_count += 1
            packet_data = {}
            
            # Packet number
            packet_data['packet_num'] = packet_count
            
            # Timestamp
            try:
                current_timestamp = float(packet.sniff_timestamp)
                packet_data['timestamp'] = current_timestamp
                
                # Calculate time delta (inter-arrival time)
                if last_timestamp is not None:
                    packet_data['time_delta'] = current_timestamp - last_timestamp
                else:
                    packet_data['time_delta'] = 0.0
                
                last_timestamp = current_timestamp
            except:
                packet_data['timestamp'] = ''
                packet_data['time_delta'] = ''
            
            # IP layer
            if hasattr(packet, 'ip'):
                packet_data['src_ip'] = packet.ip.src
                packet_data['dst_ip'] = packet.ip.dst
            else:
                packet_data['src_ip'] = ''
                packet_data['dst_ip'] = ''
            
            # TCP layer
            if hasattr(packet, 'tcp'):
                packet_data['src_port'] = packet.tcp.srcport
                packet_data['dst_port'] = packet.tcp.dstport
                packet_data['tcp_len'] = getattr(packet.tcp, 'len', '')
            else:
                packet_data['src_port'] = ''
                packet_data['dst_port'] = ''
                packet_data['tcp_len'] = ''
            
            # Modbus layer - detailed extraction for anomaly detection
            if hasattr(packet, 'modbus'):
                modbus = packet.modbus
                
                # Basic Modbus header fields
                packet_data['transaction_id'] = getattr(modbus, 'transid', '')
                packet_data['protocol_id'] = getattr(modbus, 'protid', '')
                packet_data['modbus_length'] = getattr(modbus, 'len', '')
                packet_data['unit_id'] = getattr(modbus, 'unitid', '')
                
                # Function code - critical for anomaly detection
                func_code = getattr(modbus, 'func_code', '')
                packet_data['function_code'] = func_code
                
                # Determine if response or request
                # Response function codes are typically > 0x80 for exceptions
                # or can be determined by packet direction
                try:
                    if func_code:
                        fc_int = int(str(func_code), 0)  # Handle hex strings
                        packet_data['is_exception'] = 1 if fc_int >= 0x80 else 0
                        
                        # Typically responses come from port 502
                        packet_data['is_response'] = 1 if packet_data['src_port'] == '502' else 0
                    else:
                        packet_data['is_exception'] = 0
                        packet_data['is_response'] = 0
                except:
                    packet_data['is_exception'] = 0
                    packet_data['is_response'] = 0
                
                # Exception code (for error detection)
                packet_data['exception_code'] = getattr(modbus, 'exception_code', '')
                
                # Address and count fields (important for access pattern analysis)
                packet_data['reference_num'] = getattr(modbus, 'reference_num', '')
                packet_data['word_count'] = getattr(modbus, 'word_cnt', '')
                packet_data['byte_count'] = getattr(modbus, 'byte_cnt', '')
                
                # Extract register values for value-based anomaly detection
                register_values = []
                
                # Try to get register data
                for field_name in dir(modbus):
                    if any(keyword in field_name.lower() for keyword in ['uint16', 'register', 'data']):
                        try:
                            value = getattr(modbus, field_name)
                            if value and not callable(value) and field_name[0] != '_':
                                register_values.append(str(value))
                        except:
                            pass
                
                packet_data['register_values'] = ','.join(register_values) if register_values else ''
                packet_data['num_values'] = len(register_values)
                
            else:
                # Fill with empty/default values if no Modbus layer
                for key in headers[8:]:
                    if key in ['is_response', 'is_exception', 'num_values']:
                        packet_data[key] = 0
                    else:
                        packet_data[key] = ''
            
            modbus_packets.append(packet_data)
            
            if packet_count % 100 == 0:
                print(f"Processed {packet_count} packets...")
    
    except Exception as e:
        print(f"Finished processing. Total packets: {packet_count}")
        if packet_count == 0:
            print(f"Error or no packets found: {e}")
    
    finally:
        cap.close()
    
    # Write to CSV
    print(f"\nWriting {len(modbus_packets)} Modbus packets to {output_csv}")
    
    if len(modbus_packets) > 0:
        with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=headers)
            writer.writeheader()
            writer.writerows(modbus_packets)
        
        print(f"✓ Successfully extracted {len(modbus_packets)} Modbus/TCP packets")
        print(f"\nFeatures extracted for anomaly detection:")
        print(f"  - Temporal: timestamps, inter-arrival times")
        print(f"  - Network: IPs, ports, packet sizes")
        print(f"  - Modbus: function codes, transaction IDs, unit IDs")
        print(f"  - Behavioral: request/response patterns, exceptions")
        print(f"  - Data: register values and counts")
    else:
        print("⚠ No Modbus packets found to extract")
    
    return len(modbus_packets)

In [ ]:
def check_modbus_in_pcap(pcap_file, max_packets=1000):
    """
    Quick check to see if a pcap file contains Modbus/TCP traffic.
    
    Args:
        pcap_file: Path to the pcap file
        max_packets: Maximum number of packets to check
    
    Returns:
        Boolean indicating if Modbus traffic was found
    """
    print(f"Checking for Modbus/TCP traffic in: {pcap_file}")
    
    # Try with Modbus filter
    cap = pyshark.FileCapture(pcap_file, display_filter='modbus')
    
    modbus_found = False
    count = 0
    
    try:
        for packet in cap:
            if hasattr(packet, 'modbus'):
                modbus_found = True
                print(f"✓ Modbus/TCP traffic detected!")
                print(f"  First packet at: {packet.sniff_timestamp}")
                print(f"  Source: {packet.ip.src}:{packet.tcp.srcport}")
                print(f"  Destination: {packet.ip.dst}:{packet.tcp.dstport}")
                if hasattr(packet.modbus, 'func_code'):
                    print(f"  Function Code: {packet.modbus.func_code}")
                break
            
            count += 1
            if count >= max_packets:
                break
                
    except Exception as e:
        print(f"No Modbus traffic found or error: {e}")
    
    finally:
        cap.close()
    
    if not modbus_found:
        print("✗ No Modbus/TCP traffic found in the first {} packets".format(max_packets))
    
    return modbus_found

## Features for Anomaly Detection

The extracted CSV contains these feature categories optimized for detecting anomalous Modbus/TCP traffic:

### Temporal Features (Time-based anomalies)
- **timestamp**: Absolute time of packet
- **time_delta**: Inter-arrival time between packets
  - *Useful for detecting*: Unusual timing patterns, floods, slow scans

### Network Features (Connection anomalies)
- **src_ip, dst_ip**: Source and destination IP addresses
- **src_port, dst_port**: TCP ports
- **tcp_len**: TCP payload size
  - *Useful for detecting*: Unauthorized connections, port scanning, abnormal packet sizes

### Modbus Protocol Features (Protocol anomalies)
- **transaction_id**: Modbus transaction identifier
- **protocol_id**: Should always be 0 for Modbus/TCP
- **modbus_length**: Message length
- **unit_id**: Modbus device/slave address
  - *Useful for detecting*: Malformed packets, protocol violations, unauthorized devices

### Behavioral Features (Access pattern anomalies)
- **function_code**: Modbus operation (1=Read Coils, 3=Read Holding Registers, 16=Write Multiple Registers, etc.)
- **is_response**: 1 for responses, 0 for requests
- **is_exception**: 1 for error responses
- **exception_code**: Type of error (if applicable)
  - *Useful for detecting*: Unusual operations, excessive errors, unauthorized write attempts

### Data Access Features (Data anomalies)
- **reference_num**: Starting register/coil address
- **word_count**: Number of registers/coils accessed
- **byte_count**: Data size in bytes
- **register_values**: Actual data values
- **num_values**: Count of data values
  - *Useful for detecting*: Out-of-range values, unusual access patterns, data exfiltration

In [ ]:
# Example usage:
# Specify your pcap file path
pcap_file = 'path/to/your/capture.pcap'
output_csv = 'modbus_anomaly_features.csv'

# Check for Modbus traffic first
# if check_modbus_in_pcap(pcap_file):
#     # Extract all Modbus packets with anomaly detection features
#     num_packets = extract_modbus_to_csv(pcap_file, output_csv)
#     print(f"\n✓ Created anomaly detection dataset with {num_packets} packets")